# 01 Extract, Transform, Load

**Purpose:** build reproducible BNPL tables, impute missing historical fraud scores, export completed transactions and calculate merchant metrics.

| | Details |
|---|---|
| Inputs | Supplied files in `tables/landing/`; four official ABS 2021 sources below. |
| Internal tables | `tables/raw/`, `tables/curated/`, and `tables/analysis/`; models, metrics and plots under `tables/analysis/fraud_diagnostics/`. |
| Final deliverables | `tables/analysis/transactions_imputed/transactions_imputed.csv` and `tables/analysis/transactions_imputed/transactions_imputed.parquet`. |
| Run | Run top to bottom from this repository or `code/`, in a fresh Python kernel with Java 17+, PySpark, pandas, PyArrow, requests, openpyxl, NumPy, SciPy, scikit-learn, joblib and DuckDB. |
| Reruns | Reuse cached ABS extracts; rebuild generated tables through staged writes. Supplied landing files are retained. |

Spark handles transaction joins, daily features and merchant aggregation. Small labelled datasets use scikit-learn; PyArrow streams predictions; DuckDB writes and verifies the two final files. Internal Spark tables use uncompressed Parquet; the final Parquet uses Zstandard compression.

Sections 1–10 build curated data and earlier-date spending history. Sections 11–15 fit and audit historical score estimates. Section 16 exports the completed dataset. Section 17 updates merchant metrics using that export. Curated fraud columns preserve original missingness; completed output columns contain the supplied-or-imputed scores.

If `getOrCreate()` reports a refused connection after a stopped Java process, restart the notebook kernel and run from the top. An old Python Spark object cannot reconnect to a dead JVM.


## 1. Paths, Spark and small helpers

Use the existing landing directory and separate output directories. 

`require_key` is a function that enforces one non-null row per join key, so lookup joins cannot duplicate orders. 
- e.g. `User` table cannot contain two rows for the same `user_id` or `consumer_id`
`write_table` first writes beside the destination, then replaces it with rollback on failure. Callers read the newly saved table instead of retaining obsolete file references. Consumer enrichment rebuilds its base from raw tables on reruns.


In [1]:
from pathlib import Path
from functools import reduce
from tempfile import TemporaryDirectory
from zipfile import ZipFile
import pandas as pd
import requests
from pyspark.sql import SparkSession, functions as F

# Find the repository from either its root or a nested notebook directory
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "tables/landing").is_dir()), None)
if ROOT is None:
    raise FileNotFoundError("Run from the repository or code/ directory.")
LANDING, RAW, CURATED = [ROOT / "tables" / layer for layer in ["landing", "raw", "curated"]]
ABS = LANDING / "abs_2021"
PART1 = LANDING / "project-2-bnpl-tables-part1"

# Use four local partitions; shuffle partitions control parallel work during joins and aggregations
spark = (SparkSession.builder.master("local[4]").appName("BNPL ETL")
         .config("spark.driver.memory", "2g")
         .config("spark.sql.shuffle.partitions", "32").config("spark.ui.showConsoleProgress", "false")
         .config("spark.sql.session.timeZone", "Australia/Melbourne").getOrCreate())
spark.sparkContext.setLogLevel("ERROR")

def write_table(df, folder, partition=None):
    """Replace a Parquet table safely, including when ``df`` reads that table.

    Materialise the complete replacement beside the destination before moving
    the existing table. Restore the existing table if publishing fails. Return
    a new Spark read so callers do not retain obsolete file listings.
    """
    folder = Path(folder)
    folder.parent.mkdir(parents=True, exist_ok=True)
    stage_context = TemporaryDirectory(prefix=f".{folder.name}_stage_", dir=folder.parent)
    stage = Path(stage_context.name)
    staged_data, previous = stage / "data", stage / "previous"
    cleanup_allowed = True
    try:
        writer = df.write.mode("overwrite").option("compression", "uncompressed")
        (writer.partitionBy(partition) if partition else writer).parquet(str(staged_data))
        if folder.exists():
            folder.rename(previous)
        try:
            staged_data.rename(folder)
        except BaseException:
            if previous.exists():
                try:
                    previous.rename(folder)
                except BaseException as restore_error:
                    # TemporaryDirectory otherwise removes the only surviving
                    # original during cleanup. Preserve it for recovery even if
                    # both publication and rollback fail (e.g. a disk failure).
                    cleanup_allowed = False
                    stage_context._finalizer.detach()
                    raise RuntimeError(
                        f"Could not restore {folder}; original table retained at {previous}."
                    ) from restore_error
            raise
    finally:
        if cleanup_allowed:
            stage_context.cleanup()
    df.sparkSession.catalog.refreshByPath(str(folder))
    return df.sparkSession.read.parquet(str(folder))

def require_key(df, keys, label):
    """Raise an error if any key is null or repeated, including for keys spanning several columns."""
    # Null keys cannot match; repeated keys on a lookup table multiply joined rows
    null_key = reduce(lambda a, b: a | b, [F.col(k).isNull() for k in keys])

    # Combine column checks with OR: a row is invalid if any part of its key is null
    if df.filter(null_key).limit(1).count() or df.groupBy(*keys).count().filter("count > 1").limit(1).count():
        raise ValueError(f"{label}: null or repeated key {keys}; review before joining.")

def prepare_consumer_records(users, details):
    """Rebuild consumer attributes from source tables without retaining an old income join."""
    result = (users.join(details.withColumn("consumer_record_matched", F.lit(True)),
                         "consumer_id", "left")
              .drop("consumer_id").fillna(False, subset=["consumer_record_matched"])
              .withColumn("state", F.upper(F.trim("state"))))
    return result.withColumn("postcode", F.when(F.trim("postcode").rlike(r"^\d{3,4}$"),
                             F.lpad(F.trim("postcode"), 4, "0")).otherwise(F.col("postcode")))


## 2. Landing: acquire ABS 2021 income and geographic lookups

We will use the **G02 `Median_tot_prsnl_inc_weekly`** ABS dataset for median weekly personal income in dollars at SA2 level. 

To map `postcode` (from consumer table) to `SA2` areas, we will use the smallest ABS area unit, **mesh blocks**. Decompose both Postal Area and Statistical Area 2 into mesh blocks and find their overlap to calculate a population-weighted average for district median income per consumer, based on their postcode.

- [2021 Census DataPacks](https://www.abs.gov.au/census/find-census-data/datapacks): Australia, SA2, General Community Profile, G02.
- [2021 allocation files](https://www.abs.gov.au/statistics/standards/australian-statistical-geography-standard-asgs/edition-3-july-2021-june-2026/access-and-downloads/allocation-files): mesh block → SA2/state and mesh block → Postal Area (POA).
- [2021 mesh-block counts](https://www.abs.gov.au/census/guide-census-data/mesh-block-counts/2021): usual-resident population for the weights.

Read the G02 table (selected medians and averages) from the Census ZIP for SA2 income. Preserve geographic codes as strings to retain leading zeros. All income data and geographic boundaries use 2021 data, as the census is collected every 5 years. The next iteration after 2021 is in 2026. Save the ABS extracts as Parquet in landing; reuse existing extracts on reruns.

In [2]:
# Set up external dataset access URLs
allocation_url = ("https://www.abs.gov.au/statistics/standards/australian-statistical-geography-standard-asgs/"
                  "edition-3-july-2021-june-2026/access-and-downloads/allocation-files/")
abs_urls = {
    "sa2_income": "https://www.abs.gov.au/census/find-census-data/datapacks/download/2021_GCP_SA2_for_AUS_short-header.zip",
    "mb_sa2": allocation_url + "MB_2021_AUST.xlsx",
    "mb_poa": allocation_url + "POA_2021_AUST.xlsx",
    "mb_population": "https://www.abs.gov.au/census/guide-census-data/mesh-block-counts/2021/Mesh%20Block%20Counts%2C%202021.xlsx",
}
ABS.mkdir(parents=True, exist_ok=True)

# Downloads live in a temporary directory; only Parquet source extracts are retained.
with TemporaryDirectory(prefix="bnpl_abs_") as temp:
    for name, url in abs_urls.items():
        target = ABS / name / "part-00000.parquet"
        # Reuse completed extracts so reruns avoid downloading the same ABS files.
        if target.exists():
            continue
        source = Path(temp) / (name + (".zip" if name == "sa2_income" else ".xlsx"))
        response = requests.get(url, timeout=120)
        response.raise_for_status()
        source.write_bytes(response.content)
        # Read only G02 from the Census ZIP, each dataset differs in structure and needs to be read in differently
        if name == "sa2_income":
            with ZipFile(source) as archive:
                member = next(n for n in archive.namelist() if n.endswith("G02_AUST_SA2.csv"))
                frame = pd.read_csv(archive.open(member), dtype=str)
        elif name == "mb_population":
            # Stack the population tables, skipping workbook headings and retaining valid mesh-block rows.
            with pd.ExcelFile(source) as workbook:
                sheets = [s for s in workbook.sheet_names if s.startswith("Table ")]
                frame = pd.concat([pd.read_excel(workbook, sheet_name=s, skiprows=6,
                                   usecols=["MB_CODE_2021", "Person"], dtype=str) for s in sheets], ignore_index=True)
            frame = frame[frame.MB_CODE_2021.str.fullmatch(r"\d{11}", na=False)]
        else:
            columns = (["MB_CODE_2021", "SA2_CODE_2021", "STATE_CODE_2021"] if name == "mb_sa2"
                       else ["MB_CODE_2021", "POA_CODE_2021"])
            frame = pd.read_excel(source, usecols=columns, dtype=str)
        target.parent.mkdir(parents=True, exist_ok=True)
        frame.to_parquet(target, index=False, compression=None)

## 3. Raw: read the supplied datasets
We perform an initial parse of the following data to raw layer. Keep source names and values where possible.
| Dataset | table name | Original format |
|---|---|---|
| Consumers | `tbl_consumer.csv` | CSV, pipe-delimited (`\|`) |
| Consumer fraud | `consumer_fraud_probability.csv` | CSV, comma-delimited |
| Merchant fraud | `merchant_fraud_probability.csv` | CSV, comma-delimited |
| Users | `consumer_user_details.parquet` | Parquet |
| Merchants | `tbl_merchants.parquet` | Parquet |
| Transactions | `transactions*_snapshot/` | Parquet files, partitioned by `order_datetime` |


A few things to note:
- for `merchant` table, it stores merchant ABN as index, but spark reads as column directly, so pandas `reset_index()` as performed in `00_preliminary_analysis.ipynb` is unnecessary here
- for `consumers` and `fraud` CSVs, we explicit set schemas prevent IDs/postcodes being inferred incorrectly in spark reading using `spark.read.schema(schema).csv(...)`
- for `transaction` snapshots, add `order_datetime` column (recovered from partition folder name) and combine all partitions

Schema is specified using these general rules following the column profiling from `00_preliminary_analysis.ipynb`: 
- text or categories (e.g. `name`, `address`, `gender`, `state`) $\rightarrow$ string 
- geographical information (e.g. `postcode`) $\rightarrow$ string 
- record identifiers (e.g. `consumer_id`, `user_id`, `merchant_abn`) $\rightarrow$ long 
- numeric scores (e.g. `fraud_probability`) $\rightarrow$ double 

In [3]:
def read_csv(name, schema, sep=","):
    """Read a Part 1 CSV into Spark using the supplied schema and delimiter. Stops read when malformed record encountered (FAILFAST)"""
    return (spark.read.option("header", True).option("sep", sep).option("mode", "FAILFAST") # if full parse complete, no malformed records
            .option("enforceSchema", False).schema(schema).csv(str(PART1 / name)))

# Read CSVs using specified schema and read parquets directly, to a dictionary holding separate spark dfs
raw = {
    "consumers": read_csv("tbl_consumer.csv", "name string, address string, state string, postcode string, gender string, consumer_id long", "|"),
    "users": spark.read.parquet(str(PART1 / "consumer_user_details.parquet")),
    "merchants": spark.read.parquet(str(PART1 / "tbl_merchants.parquet")),
    "consumer_fraud": read_csv("consumer_fraud_probability.csv", "user_id long, order_datetime date, fraud_probability double"),
    "merchant_fraud": read_csv("merchant_fraud_probability.csv", "merchant_abn long, order_datetime date, fraud_probability double"),
}
snapshots = sorted(LANDING.glob("project-2-bnpl-tables-part*/transactions*_snapshot"))
if not snapshots:
    raise FileNotFoundError("No transaction snapshots found in landing.")

# Read snapshot roots to get dates from partition folder names, then convert to a column with date type
transactions = [(spark.read.option("basePath", str(p)).parquet(str(p))
                 .withColumn("order_datetime", F.to_date("order_datetime"))
                 .withColumn("source_snapshot", F.lit(p.name))) for p in snapshots]

# Stack snapshots by matching column names
raw["transactions"] = transactions[0]
for snapshot in transactions[1:]:
    raw["transactions"] = raw["transactions"].unionByName(snapshot)

## 4. Raw: standardise ABS keys and save Parquet tables

The six supplied datasets from part 3 each get their own directory. ABS lookups get four additional directories. Raw transactions are partitioned by `order_month`. For downstream analysis and modelling, we will read from raw/curated layer saved outputs. 

Here, we will perform column name standardisation (especially for keys) to enable table joining later. This is the last step for raw layer.

All supplied datasets read and standardised are collected in a python dictionary `raw` as spark dataframes.

In [4]:
# Give geographic keys consistent names for joins, then add to dictionary of spark dfs as in part 3
raw["sa2_income"] = spark.read.parquet(str(ABS / "sa2_income")).select(
    F.col("SA2_CODE_2021").alias("sa2_code"),
    F.col("Median_tot_prsnl_inc_weekly").cast("double").alias("median_personal_income_weekly")) # cast to numeric
raw["mb_sa2"] = spark.read.parquet(str(ABS / "mb_sa2")).select(
    F.col("MB_CODE_2021").alias("mb_code"), F.col("SA2_CODE_2021").alias("sa2_code"),
    F.col("STATE_CODE_2021").alias("state_code"))
raw["mb_poa"] = spark.read.parquet(str(ABS / "mb_poa")).select(
    F.col("MB_CODE_2021").alias("mb_code"), F.col("POA_CODE_2021").alias("postcode"))
raw["mb_population"] = spark.read.parquet(str(ABS / "mb_population")).select(
    F.col("MB_CODE_2021").alias("mb_code"), F.col("Person").cast("long").alias("population")) # cast to numeric

# Add `order_month` for partitioning saved parquet into monthly folders
raw["transactions"] = raw["transactions"].withColumn("order_month", F.date_format("order_datetime", "yyyy-MM"))

# Save raw tables to disk as raw layer (partition by month if transactions)
for name, df in raw.items():
    write_table(df, RAW / name, "order_month" if name == "transactions" else None)

# Reload the saved raw tables so curated processing starts from a reproducible on-disk boundary
raw = {name: spark.read.parquet(str(RAW / name)) for name in raw}

## 5. Curated: replace the user bridge and prepare merchant/fraud tables

For the curated layer, we will first try to simplify joins and also deal with some join inconsistencies: 
1. For `consumers` and `users`: merge consumer details onto the user mappings and drop `consumer_id` such that the curated consumer key is now **`user_id`** used for all downstream analysis and modelling 
2. For `consumer`: zero-pad postcodes to valid 4-digit Australian postcodes
3. For `merchants`: parse merchant `tags` into category, letter group and take rate. Keep take rates and fraud scores in their supplied units for now. 
4. For `fraud`: remove exact duplicate score rows, throw error if any conflicting scores for a single key = (consumer/merchant id, order datetime)

In [5]:
# Validate user-consumer mapping and one-row-per-order assumption using defined `require_key` function
for name, keys in [("consumers", ["consumer_id"]), ("users", ["user_id"]), ("users", ["consumer_id"]),
                   ("merchants", ["merchant_abn"]), ("transactions", ["order_id"])]:
    require_key(raw[name], keys, name)

# Users table contains `user_id` which joins with all other tables
consumers = prepare_consumer_records(raw["users"], raw["consumers"])

# Confirm every user has a matching consumer record
unmatched = consumers.filter(~F.col("consumer_record_matched")).count()
print(f"Users without consumer details: {unmatched:,}")

# Confirm no consumers were left out of the user-based curated table
unmapped = raw["consumers"].join(raw["users"], "consumer_id", "left_anti").count()
print(f"Consumers without a user mapping: {unmapped:,}")

Users without consumer details: 0
Consumers without a user mapping: 0


In [6]:
# For merchant table, parse tag into category / letter / take-rate fields
pattern = r"(?i)^[\[(]{2}(.*?)[\])]\s*,\s*[\[(]([A-Za-z])[\])]\s*,\s*[\[(]take rate:\s*([0-9]+(?:\.[0-9]+)?)[\])]{2}$" # regex pattern to extract each tag component
merchants = (raw["merchants"]
    .withColumn("category", F.lower(F.trim(F.regexp_extract("tags", pattern, 1))))
    .withColumn("letter_group", F.lower(F.regexp_extract("tags", pattern, 2)))
    .withColumn("take_rate", F.when(F.regexp_extract("tags", pattern, 3) != "", F.regexp_extract("tags", pattern, 3).cast("double"))))

# Match the preliminary category rule: collapse whitespace and standardise comma/colon/hyphen spacing.
merchants = merchants.withColumn("category", F.trim(F.regexp_replace(
    F.regexp_replace(F.regexp_replace("category", r"\s+", " "), r"\s*([,:])\s*", "$1 "), r"\s*-\s*", " - ")))

# For merchant table, sanity check for tag parse
if merchants.filter(F.col("take_rate").isNull()).limit(1).count():
    raise ValueError("Some merchant tags did not parse; inspect them before continuing.")

# For fraud tables, remove identical score rows
consumer_fraud = raw["consumer_fraud"].dropDuplicates().withColumnRenamed("fraud_probability", "consumer_fraud_probability")
merchant_fraud = raw["merchant_fraud"].dropDuplicates().withColumnRenamed("fraud_probability", "merchant_fraud_probability")
require_key(consumer_fraud, ["user_id", "order_datetime"], "consumer fraud")

# For fraud tables, fail if conflicting scores for same key = (consumer/merchant, datetime))
require_key(merchant_fraud, ["merchant_abn", "order_datetime"], "merchant fraud")

## 6. Curated: postcode-SA2 median income lookup

`Consumers` table have state/postcode, not SA2. [ABS Postal Areas](https://www.abs.gov.au/statistics/standards/australian-statistical-geography-standard-asgs/edition-3-july-2021-june-2026/non-abs-structures/postal-areas) approximate postcodes and may cross state borders, which we will adjust using consumer given state. We will use mesh blocks (smallest ABS geographical unit) to establish state-postcode overlap with SA2, weighting each overlap by its usual-resident population. This is a derived mapping using public ABS data.

For each state/postcode area $p$, we calculate the estimated (median) weekly person income `district_income`: 
$$\widehat{I}_p
=
\frac{\displaystyle\sum_{s \in S_p} N_{ps}\,m_s}
     {\displaystyle\sum_{s \in S_p} N_{ps}}$$
- $S_p$: SA2 areas overlapping postcode area $p$ with usable income data
- $N_{ps}$: population living in the overlap between postcode area $p$ and SA2 $s$
- $m_s$: median weekly personal income in SA2 $s$, in dollars
- $\widehat{I}_p$: population-weighted average of those SA2 medians. NULL if population is zero. 

This is an **average of SA2 medians**, to be joined with individual consumers, NOT the postcode median! Population is an approximated geographic weight, as personal income is for people aged 15+ only. Exclude non-geographic ABS codes from the income lookup such as `9494` (no usual address), `9797` (migratory/offshore/shipping) and `ZZZZ` (outside Australia).

Create table with calculated (weighted) median income per state/postcode to join to consumers. Note the following join assumptions:
- For each postcode, median income left as NULL if all its mapped SA2 zones have NULL income or zero total population 
     - as justified in `00_preliminary_analysis.ipynb`, we will also exclude the 42 SA2 zones with 0-income due to data ambiguity (and would skew the data if averaged)
- For each postcode, record number of mapped SA2 zones and observed population
- Assignment specifies SA2 datasets from ABS, otherwise we might consider using direct POA income table...

The created dataframes are of the following structure:
| Table | Key | Contains |
|---|---|---|
| `mesh` | `mb_code` | Population per mesh block |
| `overlap` | `state`, `postcode`, `sa2_code` | Population per state–POA–SA2 overlap |
| `overlap_income` | `state`, `postcode`, `sa2_code` | Overlap population and SA2 median income |
| `postcode_income` | `state`, `postcode` | Population and weighted income estimate per state–POA |

For `postcode_income`, the columns are: 
- `income_sa2_count`: number of sa2 zones overlapping with this state-poa
- `area_population`: population per state-poa (all)
- `income_population`: population per state-poa with nonzero median income
- `district_income`: weighted median income avg per state-poa
- `income_coverage`: proportion of state-poa with nonzero median income

In [7]:
# Check ABS data tables contain required keys
for name, keys in [("sa2_income", ["sa2_code"]), ("mb_sa2", ["mb_code"]),
                   ("mb_poa", ["mb_code"]), ("mb_population", ["mb_code"])]:
    require_key(raw[name], keys, name) 

# State code mapping between ABS `state_code_2021` and consumers `state`
state_codes = {"1": "NSW", "2": "VIC", "3": "QLD", "4": "SA", "5": "WA", "6": "TAS", "7": "NT", "8": "ACT", "9": "OT"}
state_map = F.create_map(*[F.lit(v) for pair in state_codes.items() for v in pair])

# Mesh blocks dataframe: one mesh per row, with SA2, postcode, population, state abbreviation   
mesh = (raw["mb_sa2"].join(raw["mb_poa"], "mb_code", "left")
        .join(raw["mb_population"], "mb_code", "left")
        .withColumn("state", state_map[F.col("state_code")]))

# Identify all SA2s overlapping each state/postcode and sums their overlap population
overlap = (mesh.filter(F.col("postcode").rlike(r"^\d{4}$")       # exclude irregular format codes
                       & ~F.col("postcode").isin("9494", "9797") # exclude non-graphic POA codes
                       & (F.col("population") > 0))              # exclude unpopulated blocks 
           .groupBy("state", "postcode", "sa2_code").agg(F.sum("population").alias("population"))) # population per (state)-poa-sa2

# Validate each row with (state)-poa-sa2 key
require_key(overlap, ["state", "postcode", "sa2_code"], "postcode-SA2 overlaps")
write_table(overlap, CURATED / "postcode_sa2_population")

# Attach per-sa2 income to overlap (state-poa-sa2) regions
overlap_income = overlap.join(raw["sa2_income"], "sa2_code", "left")

# Calculate weighted average per state-poa: weighted income sum divided by population with usable income
postcode_income = (overlap_income.groupBy("state", "postcode").agg( # new key
    F.countDistinct("sa2_code").alias("income_sa2_count"), 
    F.sum("population").alias("area_population"), # postcode population
    F.sum(F.when(F.col("median_personal_income_weekly") > 0, 
                 F.col("population")).otherwise(0)).alias("income_population"), # state-poa population, summed from state-poa-sa2 populations with positive median income
    F.sum(F.when(F.col("median_personal_income_weekly") > 0, 
                 F.col("population") * F.col("median_personal_income_weekly"))).alias("income_sum")) # weighted income sum per state-poa when income is postive
    .withColumn("district_income", F.when(F.col("income_population") > 0, 
                                          F.col("income_sum") / F.col("income_population"))) # weighted income avg per state-poa when usable population is non-zero
    .withColumn("income_coverage", F.col("income_population") / F.col("area_population")) # proportion of state-poa with positive income 
    .drop("income_sum")) # drop intermediate column

# Validate each row with state-poa key
require_key(postcode_income, ["state", "postcode"], "postcode income")
write_table(postcode_income, CURATED / "postcode_income")

## 7. Curated: join calculated median income to consumers

Left-join calculated median income lookup table on **state + postcode**, preserving every user mapping. Missing income stays null. 
- `income_matched`: TRUE if an estimate is available
- `income_coverage`: whether the weighted median income is calculated from all resident population in the state/POA overlap

In [8]:
# Read from postcode income lookup
postcode_income = spark.read.parquet(str(CURATED / "postcode_income"))

# Broadcast the small postcode lookup to each partition to avoid shuffle
income_lookup = postcode_income.drop("area_population", "income_population")

# Rebuild from saved raw tables on each rerun; do not reuse a previous income-join lineage.
consumers = prepare_consumer_records(
    spark.read.parquet(str(RAW / "users")), spark.read.parquet(str(RAW / "consumers")))
consumers = (consumers.join(F.broadcast(income_lookup), ["state", "postcode"], "left") # left join keeps users with no income match
             .withColumn("income_matched", F.col("district_income").isNotNull())) # flag if no match

# Validate each row with consumer key 
require_key(consumers, ["user_id"], "curated consumers")

# Save curated layer reference tables
curated = {"consumers": consumers, "merchants": merchants, "consumer_fraud": consumer_fraud, "merchant_fraud": merchant_fraud}
for name, df in curated.items():
    write_table(df, CURATED / name)
curated = {name: spark.read.parquet(str(CURATED / name)) for name in curated}

## 8. Curated: enrich transactions

Keep **one row per order** with the diagram's five additional attributes: consumer/merchant fraud probabilities, consumer postcode, gender and district income. Merchant name/tags stay in the separate curated merchant table, linked by ABN.

Joining logic: 
- Join entity fraud scores directly on **entity ID + order date**, independently of whether entity details exist. Entity is either consumer or merchant.
- Left joins keep unmatched orders and null scores.
- `merchant_record_matched` and `consumer_record_matched` distinguish missing master records. 
- `income_matched` identifies income lookup matches. Detailed coverage stays in the curated consumer/postcode tables; the income source year is 2021. 

In [9]:
# Only select consumer attributes reuqired for each order
consumer_attributes = curated["consumers"].select(
    "user_id", 
    F.col("postcode").alias("consumer_postcode"), 
    F.col("gender").alias("consumer_gender"),
    F.col("district_income").alias("consumer_district_income"), 
    "consumer_record_matched",
    "income_matched", 
    "income_coverage")

# Check if merchant exists in merchant table (consumers checked earlier in section 5)
merchant_keys = curated["merchants"].select("merchant_abn").withColumn("merchant_record_matched", F.lit(True))

# Join required columns (merchant and fraud score lookups are broadcasted)
transactions = (raw["transactions"].join(consumer_attributes, "user_id", "left") 
                .join(F.broadcast(merchant_keys), "merchant_abn", "left")
                .join(F.broadcast(curated["consumer_fraud"]), ["user_id", "order_datetime"], "left") # requires exact entity-date match
                .join(F.broadcast(curated["merchant_fraud"]), ["merchant_abn", "order_datetime"], "left")
                .fillna(False, subset=["consumer_record_matched", "merchant_record_matched", "income_matched"])) # flag false if unknown score

# Select necessary columns and save to disk
transactions = transactions.select("order_id", "order_datetime", "order_month", "dollar_value", 
                                   "user_id", "merchant_abn", "consumer_fraud_probability", "merchant_fraud_probability",
                                   "consumer_postcode", "consumer_gender", "consumer_district_income", 
                                   "income_matched", "consumer_record_matched", "merchant_record_matched")

write_table(transactions, CURATED / "transactions", "order_month")

## 9. Verify row preservation

Check the saved result against raw that all rows have been preserved appropriately.

In [10]:
# Compare persisted outputs to catch any rows lost or multiplied by the enrichment joins
raw_count = raw["transactions"].count()
curated_count = spark.read.parquet(str(CURATED / "transactions")).count()
if curated_count != raw_count:
    raise ValueError(f"Transaction count changed: {raw_count:,} raw → {curated_count:,} curated.")
print(f"Saved raw and curated Parquet tables; retained {curated_count:,} transactions.")

Saved raw and curated Parquet tables; retained 14,195,505 transactions.


## 10. Some `transactions` statistics
Check the data in `transations`, especially orders with missing important attributes.

In [11]:
transaction_total = transactions.count()
transaction_unknown_consumer_fraud = transactions.where(F.col('consumer_fraud_probability').isNull()).count()
transaction_unknown_merchant_fraud = transactions.where(F.col('merchant_fraud_probability').isNull()).count()
transaction_unknown_or_fraud = transactions.where((F.col('consumer_fraud_probability').isNull()) | (F.col('merchant_fraud_probability').isNull())).count()
transaction_unknown_and_fraud = transactions.where((F.col('consumer_fraud_probability').isNull()) & (F.col('merchant_fraud_probability').isNull())).count()

print(f"# Orders in total: {transaction_total}")
print(f"# Orders missing consumer_fraud probability: {transaction_unknown_consumer_fraud} ({transaction_unknown_consumer_fraud/transaction_total:.4%})")
print(f"# Orders missing merchant_fraud probability: {transaction_unknown_merchant_fraud} ({transaction_unknown_merchant_fraud/transaction_total:.4%})")
print(f"# Orders missing either fraud probability: {transaction_unknown_or_fraud} ({transaction_unknown_or_fraud/transaction_total:.4%})")
print(f"# Orders missing both fraud probability: {transaction_unknown_and_fraud} ({transaction_unknown_and_fraud/transaction_total:.4%})")

# Orders in total: 14195505
# Orders missing consumer_fraud probability: 14115157 (99.4340%)
# Orders missing merchant_fraud probability: 14191446 (99.9714%)
# Orders missing either fraud probability: 14195100 (99.9971%)
# Orders missing both fraud probability: 14111503 (99.4082%)


In [12]:
print(transactions.where(F.col("order_id").isNull()).count())
print(transactions.where(F.col("order_datetime").isNull()).count())
print(transactions.where(F.col("order_month").isNull()).count())
print(transactions.where(F.col("dollar_value").isNull()).count())
print(transactions.where(F.col("user_id").isNull()).count())
print(transactions.where(F.col("merchant_abn").isNull()).count())
print(transactions.where(F.col("consumer_fraud_probability").isNull()).count())
print(transactions.where(F.col("merchant_fraud_probability").isNull()).count())
print(transactions.where(F.col("consumer_postcode").isNull()).count())
print(transactions.where(F.col("consumer_gender").isNull()).count())
print(transactions.where(F.col("consumer_district_income").isNull()).count())

0
0
0
0
0
0
14115157
14191446
0
0
2368565


In [13]:
transactions.select(
    F.min("consumer_district_income"), 
    F.max("consumer_district_income")
    ).show()

+-----------------------------+-----------------------------+
|min(consumer_district_income)|max(consumer_district_income)|
+-----------------------------+-----------------------------+
|                        276.0|                       3250.0|
+-----------------------------+-----------------------------+



## Curated-stage interpretation and spending enrichment

The income field is a population-weighted average of positive SA2 median weekly personal incomes, not an individual income or a postcode median. The postcode approximation, all-age weights and zero-income rule are project choices. The enrichment is retrospective because 2021 Census results were released later.

Curated fraud columns preserve original supplied scores and missingness. Sections 11–16 impute historical gaps and export the completed dataset. Scores and merchant take rates use percentage units; section 17 converts them when estimating fee revenue. No `is_fraud` ground-truth label is created.

The next cell fills missing district income with the approximate median available transaction-level income. It also calculates consumer average spending across strictly earlier calendar dates, signed `spending_diff`, and `absolute_spending_diff`. All first-date orders have no history; section 16 supplies their explicitly flagged median fallback. Order IDs are never used to infer within-day purchase times.

Curated directories are `consumers`, `merchants`, `consumer_fraud`, `merchant_fraud`, `postcode_sa2_population`, `postcode_income`, and `transactions`. Raw retains the users bridge; curated consumers are keyed by user ID. Unknown merchants remain in transactions and are reported in section 17.


In [14]:
from pyspark.sql.window import Window


def enrich_transaction_spending(orders):
    """Fill missing district income and add spending based on earlier dates.

    All orders on a consumer's first observed date have no spending history.
    Orders on the same date share the average over that consumer's earlier
    dates; order IDs do not establish purchase times. ``spending_diff`` is the
    signed current value minus that historical average.
    """
    orders = orders.drop("consumer_avg_spending", "spending_diff", "absolute_spending_diff")
    medians = orders.approxQuantile("consumer_district_income", [0.5], 0.001)
    if not medians:
        raise ValueError("Cannot impute district income: no observed income values.")
    orders = orders.withColumn(
        "consumer_district_income",
        F.coalesce(F.col("consumer_district_income"), F.lit(medians[0])),
    )
    daily = orders.groupBy("user_id", "order_datetime").agg(
        F.sum("dollar_value").alias("_day_spending"),
        F.count("*").alias("_day_orders"),
    )
    previous_dates = (
        Window.partitionBy("user_id")
        .orderBy("order_datetime")
        .rowsBetween(Window.unboundedPreceding, -1)
    )
    history = (
        daily.withColumn("_prior_spending", F.sum("_day_spending").over(previous_dates))
        .withColumn("_prior_orders", F.sum("_day_orders").over(previous_dates))
        .select(
            "user_id",
            "order_datetime",
            F.when(
                F.col("_prior_orders") > 0,
                F.col("_prior_spending") / F.col("_prior_orders"),
            ).alias("consumer_avg_spending"),
        )
    )
    return (
        orders.join(history, ["user_id", "order_datetime"], "left")
        .withColumn("spending_diff", F.col("dollar_value") - F.col("consumer_avg_spending"))
        .withColumn("absolute_spending_diff", F.abs("spending_diff"))
    )


transactions_path = CURATED / "transactions"
transactions_to_enrich = spark.read.parquet(str(transactions_path))
transactions = write_table(enrich_transaction_spending(transactions_to_enrich), transactions_path, "order_month")


In [15]:
# Preview the persisted earlier-date definition used throughout this notebook.
transactions_history = spark.read.parquet(str(CURATED / "transactions"))
transactions_history.select("user_id", "order_datetime", "order_id", "dollar_value",
                            "consumer_avg_spending", "spending_diff", "absolute_spending_diff") \
    .orderBy("user_id", "order_datetime", "order_id").show(5, truncate=False)


+-------+--------------+------------------------------------+------------------+---------------------+-------------------+----------------------+
|user_id|order_datetime|order_id                            |dollar_value      |consumer_avg_spending|spending_diff      |absolute_spending_diff|
+-------+--------------+------------------------------------+------------------+---------------------+-------------------+----------------------+
|1      |2021-02-28    |0c37b3f7-c7f1-48cb-bcc7-0a58e76608ea|133.22689421562643|NULL                 |NULL               |NULL                  |
|1      |2021-02-28    |40a2ff69-ea34-4657-8429-df7ca957d6a1|30.441348317517228|NULL                 |NULL               |NULL                  |
|1      |2021-03-02    |5b6c059c-6831-4710-a2ff-76e265b4a509|301.67749826065744|81.83412126657183    |219.84337699408562 |219.84337699408562    |
|1      |2021-03-04    |39e028ec-1663-4aff-a341-a9ad882d68d7|5.904950671999753 |155.11524693126702   |-149.21029625926727|14

In [16]:
transactions_df = spark.read.parquet(str(CURATED / "transactions"))
null_counts_df = transactions_df.select([
    F.sum(F.col(column).isNull().cast("long")).alias(column)
    for column in transactions_df.columns
])
null_counts_df.show(truncate=False)

+-------+--------------+--------+------------+------------+--------------------------+--------------------------+-----------------+---------------+------------------------+--------------+-----------------------+-----------------------+---------------------+-------------+----------------------+-----------+
|user_id|order_datetime|order_id|dollar_value|merchant_abn|consumer_fraud_probability|merchant_fraud_probability|consumer_postcode|consumer_gender|consumer_district_income|income_matched|consumer_record_matched|merchant_record_matched|consumer_avg_spending|spending_diff|absolute_spending_diff|order_month|
+-------+--------------+--------+------------+------------+--------------------------+--------------------------+-----------------+---------------+------------------------+--------------+-----------------------+-----------------------+---------------------+-------------+----------------------+-----------+
|0      |0             |0       |0           |0           |14115157            

## 11. Daily fraud-score features

The objective is to fill missing historical customer/date and merchant/date scores as accurately as the observed labels support. This stage adds consumer **ordinary least-squares regression** and a small pooled merchant **Gaussian process**. Requires scikit-learn, SciPy, NumPy, PyArrow and joblib in addition to the ETL dependencies. Run sections 11–15 after the curated tables exist; to rerun these stages against existing curated data, first run section 1 for Spark and the shared table writer.

One target row is one **entity/date**, not one order or one permanent score per entity. Same-day totals include all completed transactions for that entity/date; average order value is total spending divided by order count. The consumer uses same-day spending, average order value, squared spending and squared order count. The merchant uses same-day spending, order count, average order value and the number of orders on the previous 30 calendar days. Entity IDs are join keys, not model inputs.

Earlier-day features are retained only for the separate `purchase_time` models and spending-history columns. The feature/prediction tables live in `tables/analysis/`; supplied curated scores remain unchanged.

In [17]:
from pathlib import Path
from pyspark.sql import SparkSession, functions as F
from pyspark.sql.window import Window

root = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / "tables/curated/transactions").is_dir()), None)
if root is None:
    raise FileNotFoundError("Complete the curated ETL and run from the repository or code/ directory.")
spark = (SparkSession.builder.master("local[4]").appName("BNPL fraud score imputation")
         .config("spark.sql.shuffle.partitions", "32")
         .config("spark.ui.showConsoleProgress", "false").getOrCreate())
spark.sparkContext.setLogLevel("ERROR")
ANALYSIS = root / "tables/analysis"
MODEL_OUTPUT = root / "tables/analysis/fraud_diagnostics"
ANALYSIS.mkdir(parents=True, exist_ok=True)
MODEL_OUTPUT.mkdir(parents=True, exist_ok=True)


def build_score_features(orders, labels, entity, observation_start):
    """Build unique entity/date rows with same-day activity and separate earlier history."""
    key = "user_id" if entity == "consumer" else "merchant_abn"
    target = f"{entity}_fraud_probability"
    daily = orders.groupBy(key, "order_datetime").agg(
        F.sum("dollar_value").alias("day_spend"), F.count("*").alias("day_orders"))
    dates = daily.select(key, "order_datetime").unionByName(
        labels.select(key, "order_datetime")).distinct()
    base = (dates.join(daily, [key, "order_datetime"], "left")
            .fillna(0, subset=["day_spend", "day_orders"])
            .withColumn("same_day_spending", F.col("day_spend"))
            .withColumn("same_day_order_count", F.col("day_orders"))
            .withColumn("same_day_average_order_value", F.when(F.col("day_orders") > 0,
                        F.col("day_spend") / F.col("day_orders")).otherwise(0.0))
            .withColumn("same_day_activity_available", F.col("day_orders") > 0))
    history = (Window.partitionBy(key).orderBy("order_datetime")
               .rowsBetween(Window.unboundedPreceding, -1))
    base = (base.withColumn("prior_order_count", F.coalesce(F.sum("day_orders").over(history), F.lit(0)))
            .withColumn("prior_total_spend", F.coalesce(F.sum("day_spend").over(history), F.lit(0.0)))
            .withColumn("prior_avg_spend", F.when(F.col("prior_order_count") > 0,
                                                 F.col("prior_total_spend") / F.col("prior_order_count"))))
    if entity == "consumer":
        previous = daily.select(
            key, F.date_add("order_datetime", 1).alias("order_datetime"),
            F.col("day_spend").alias("previous_day_spending"),
            F.col("day_orders").alias("previous_day_order_count"),
            (F.col("day_spend") / F.col("day_orders")).alias("previous_day_average_order_value"))
        base = (base.join(previous, [key, "order_datetime"], "left")
                .fillna(0, subset=["previous_day_spending", "previous_day_order_count",
                                  "previous_day_average_order_value"])
                .withColumn("had_previous_day_orders", (F.col("previous_day_order_count") > 0).cast("double"))
                .withColumn("history_available", F.col("order_datetime") > F.lit(observation_start)))
        inputs = ["previous_day_spending", "previous_day_average_order_value",
                  "previous_day_order_count", "had_previous_day_orders"]
    elif entity == "merchant":
        days30 = (Window.partitionBy(key)
                  .orderBy(F.datediff("order_datetime", F.lit("1970-01-01"))).rangeBetween(-30, -1))
        last_order = F.max(F.when(F.col("day_orders") > 0, F.col("order_datetime"))).over(history)
        base = (base.withColumn("prior_30d_spend", F.coalesce(F.sum("day_spend").over(days30), F.lit(0.0)))
                .withColumn("prior_30d_order_count", F.coalesce(F.sum("day_orders").over(days30), F.lit(0)))
                .withColumn("days_since_last_order", F.datediff(F.col("order_datetime"), last_order))
                .withColumn("history_available", F.col("prior_order_count") > 0))
        inputs = ["prior_30d_spend", "prior_30d_order_count", "days_since_last_order"]
    else:
        raise ValueError("entity must be consumer or merchant")
    return (base.join(labels.select(key, "order_datetime", F.col(target).alias("target_score")),
                      [key, "order_datetime"], "left")
            .select(key, "order_datetime", "target_score", "history_available",
                    "same_day_spending", "same_day_average_order_value",
                    "same_day_order_count", "same_day_activity_available",
                    "prior_order_count", "prior_avg_spend", *inputs)
            .withColumn("observed_history_days", F.datediff("order_datetime", F.lit(observation_start))))


def check_score_labels(labels, entity):
    """Reject missing keys, invalid scores and duplicate entity/date targets before joining."""
    key = "user_id" if entity == "consumer" else "merchant_abn"
    target = f"{entity}_fraud_probability"
    invalid = (F.col(key).isNull() | F.col("order_datetime").isNull() | F.col(target).isNull()
               | F.isnan(target) | ~F.col(target).between(0, 100))
    if labels.where(invalid).limit(1).count():
        raise ValueError(f"Invalid {entity} fraud labels: expected unique keys and finite 0–100 scores.")
    if labels.groupBy(key, "order_datetime").count().where("count > 1").limit(1).count():
        raise ValueError(f"Duplicate {entity}/date scores; run the curated deduplication first.")


score_orders = spark.read.parquet(str(root / "tables/curated/transactions")).select(
    "order_id", "user_id", "merchant_abn", "order_datetime", "dollar_value")
observation_start = score_orders.agg(F.min("order_datetime")).first()[0]
if observation_start is None:
    raise ValueError("No dated transactions to model.")
feature_paths = {}
for entity in ["consumer", "merchant"]:
    labels = spark.read.parquet(str(root / f"tables/curated/{entity}_fraud"))
    check_score_labels(labels, entity)
    features = build_score_features(score_orders, labels, entity, observation_start)
    feature_paths[entity] = ANALYSIS / f"{entity}_score_features"
    write_table(features, feature_paths[entity])
    print(f"Saved {entity} features: {feature_paths[entity]}")


Saved consumer features: /Users/trollinmaster/Documents/Uni/Coding/VSCode/MAST30034/industry-project-group-19/tables/analysis/consumer_score_features
Saved merchant features: /Users/trollinmaster/Documents/Uni/Coding/VSCode/MAST30034/industry-project-group-19/tables/analysis/merchant_score_features


## 12. Consumer linear regression and merchant Gaussian process

The retrospective consumer equation is `score = intercept + b1*S + b2*A + b3*S² + b4*N²`, where S is same-day total spending, A is average order value and N is order count. Construct the **raw squares before scaling** the four columns; scaling is fitted on the training rows. Raw predictions are retained and delivered scores are clipped to 0–100.

The merchant GP uses four log-transformed/scaled predictors with Matérn smoothness 2.5 plus observation noise. Smoothness tuning in `02_analysis.ipynb` compared 0.5, 1.5 and 2.5 and retained 2.5. It models logit-transformed scores and returns logistic-normal score means and conditional 95% intervals. Same-day activity plus previous-30-day count was selected using validation RMSE among three small specifications; the small validation sample limits confidence in that selection. That earlier feature-selection experiment is recorded in `tables/analysis/fraud_diagnostics/merchant_feature_selection.json`; this notebook fits the chosen specification rather than rerunning that experiment.

Rows without the activity needed by a model receive its training-score mean and a low-support flag. Predictions outside observed training-feature ranges are also flagged; the GP additionally flags predictors far from its training examples. A high R² on supplied scores does not establish accuracy for missing-score rows outside that observed range. No training labels means no prediction. These models estimate supplied 0–100 scores, not verified transaction fraud outcomes.

In [18]:
"""Small score regressors; callers control label availability and time splits.

These models estimate supplied entity/date scores, not verified fraud outcomes.
They do not establish that a source score was published at purchase time.
"""
import warnings

import numpy as np
import pandas as pd
from scipy.special import expit, logit
from sklearn.exceptions import ConvergenceWarning
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import ConstantKernel, Matern, WhiteKernel
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression
from sklearn.neighbors import NearestNeighbors
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

SCORE_FEATURES = {
    "retrospective": {
        "consumer": ["same_day_spending", "same_day_average_order_value", "same_day_order_count"],
        "merchant": ["same_day_spending", "same_day_order_count",
                     "same_day_average_order_value", "prior_30d_order_count"],
    },
    "purchase_time": {
        "consumer": ["previous_day_spending", "previous_day_average_order_value",
                     "previous_day_order_count", "had_previous_day_orders"],
        "merchant": ["prior_30d_spend", "prior_30d_order_count", "prior_avg_spend",
                     "days_since_last_order"],
    },
}
_QUADRATURE_NODES, _QUADRATURE_WEIGHTS = np.polynomial.hermite.hermgauss(20)


def _availability_mask(frame, column):
    """Use same-day activity for imputation; earlier history for purchase-time models."""
    return frame[column].fillna(False).to_numpy(dtype=bool)


def _score_inputs(frame, entity, feature_scope):
    """Return numeric predictors only; identifiers and scores never enter X."""
    values = frame[SCORE_FEATURES[feature_scope][entity]].astype(float).to_numpy(copy=True)
    if np.isinf(values).any() or (values < 0).any():
        raise ValueError("Score predictors must be nonnegative, finite or missing.")
    if entity == "consumer":
        no_orders = values[:, 2] == 0
        values[no_orders, 1] = 0.0  # Average is undefined for a known empty day.
        if not np.isfinite(values).all():
            raise ValueError("Available consumer activity must have complete predictors.")
        if feature_scope == "purchase_time" and not np.isin(values[:, 3], [0, 1]).all():
            raise ValueError("had_previous_day_orders must be zero or one.")
        return values
    return np.log1p(values)


def _consumer_design(values, feature_scope):
    """Create raw squared terms BEFORE scaling, preserving the exact requested model."""
    if feature_scope == "retrospective":
        return np.column_stack([values[:, 0], values[:, 1], values[:, 0] ** 2, values[:, 2] ** 2])
    return np.column_stack([values, values[:, 0] ** 2, values[:, 2] ** 2])


def _training_ranges(values):
    """Missing-only columns have unknown bounds rather than invented ranges."""
    lower = np.full(values.shape[1], np.nan)
    upper = np.full(values.shape[1], np.nan)
    for column in range(values.shape[1]):
        observed = values[np.isfinite(values[:, column]), column]
        if len(observed):
            lower[column], upper[column] = observed.min(), observed.max()
    return lower, upper


def _outside_ranges(values, lower, upper):
    """Flag incomplete predictors or values outside fitted feature ranges."""
    unknown = ~np.isfinite(values) | ~np.isfinite(lower) | ~np.isfinite(upper)
    return (unknown | (values < lower) | (values > upper)).any(axis=1)


def _logistic_normal_summary(location, scale):
    """Mean/SD by 20-point quadrature; central conditional 95% quantiles."""
    samples = expit(location[:, None] + np.sqrt(2) * scale[:, None] * _QUADRATURE_NODES)
    weights = _QUADRATURE_WEIGHTS / np.sqrt(np.pi)
    mean = samples @ weights
    variance = np.maximum((samples ** 2) @ weights - mean ** 2, 0)
    return (100 * mean, 100 * np.sqrt(variance),
            100 * expit(location - 1.959963984540054 * scale),
            100 * expit(location + 1.959963984540054 * scale))


def fit_score_model(frame, entity, feature_scope="retrospective"):
    """Fit an OLS/GP bundle from already time-filtered labeled entity/date rows.

    All supplied labels determine the mean fallback. Rows with available
    activity teach the regressors. Fewer than 20 consumer or 10 merchant usable rows
    select the mean baseline; no labels select an unavailable bundle.
    """
    if feature_scope not in SCORE_FEATURES:
        raise ValueError("feature_scope must be 'retrospective' or 'purchase_time'.")
    if entity not in SCORE_FEATURES[feature_scope]:
        raise ValueError("entity must be 'consumer' or 'merchant'.")
    availability = "same_day_activity_available" if feature_scope == "retrospective" else "history_available"
    bundle = {"entity": entity, "feature_scope": feature_scope, "availability_column": availability,
              "kind": "unavailable", "n_labels": len(frame),
              "n_fit_rows": 0, "mean_score": np.nan,
              "feature_columns": SCORE_FEATURES[feature_scope][entity].copy(), "fit_warnings": []}
    if frame.empty:
        return bundle
    targets = frame["target_score"].to_numpy(dtype=float)
    if not (np.isfinite(targets).all() and ((targets >= 0) & (targets <= 100)).all()):
        raise ValueError("Training rows must have supplied scores within 0 to 100.")
    history = _availability_mask(frame, availability)
    bundle.update(kind="mean", mean_score=float(targets.mean()),
                  n_fit_rows=int(history.sum()))
    minimum = 20 if entity == "consumer" else 10
    if history.sum() < minimum:
        return bundle
    values = _score_inputs(frame.loc[history], entity, feature_scope)
    labels = targets[history]
    bundle["feature_lower"], bundle["feature_upper"] = _training_ranges(values)
    if entity == "consumer":
        design = _consumer_design(values, feature_scope)
        prep = StandardScaler().fit(design)
        model = LinearRegression().fit(prep.transform(design), labels)
        columns = (["same_day_spending", "same_day_average_order_value",
                    "same_day_spending_squared", "same_day_order_count_squared"]
                   if feature_scope == "retrospective" else
                   [*SCORE_FEATURES[feature_scope][entity], "previous_day_spending_squared",
                    "previous_day_order_count_squared"])
        bundle.update(kind="ols", preprocessor=prep, model=model,
                      design_columns=columns)
    else:
        prep = make_pipeline(SimpleImputer(strategy="median", keep_empty_features=True),
                             StandardScaler()).fit(values)
        transformed = prep.transform(values)
        kernel = (ConstantKernel(1.0, (1e-3, 1e3))
                  * Matern(length_scale=1.0, length_scale_bounds=(1e-2, 1e3), nu=2.5)
                  + WhiteKernel(noise_level=0.1, noise_level_bounds=(1e-5, 10)))
        model = GaussianProcessRegressor(kernel=kernel, normalize_y=True, alpha=1e-8,
                                         n_restarts_optimizer=0, random_state=19)
        with warnings.catch_warnings(record=True) as caught:
            warnings.simplefilter("always", ConvergenceWarning)
            model.fit(transformed, logit(np.clip(labels / 100, 1e-4, 1 - 1e-4)))
        neighbours = NearestNeighbors(n_neighbors=1).fit(transformed)
        # Omit X so each fitted row is excluded from its own nearest neighbour.
        threshold = float(np.quantile(neighbours.kneighbors()[0][:, 0], 0.95))
        bundle.update(kind="gp", preprocessor=prep, model=model, neighbours=neighbours,
                      support_distance_threshold=threshold,
                      fit_warnings=[str(warning.message) for warning in caught])
    return bundle


def predict_score_model(bundle, frame):
    """Return index-aligned estimates; retain raw OLS estimates and bound to 0–100.

    GP intervals are conditional model quantiles, not validated error guarantees.
    Mean fallbacks have no estimated interval and always receive low_support.
    """
    result = pd.DataFrame(index=frame.index)
    for column in ["predicted_score", "raw_prediction", "interval_low", "interval_high",
                   "predicted_score_sd", "support_distance"]:
        result[column] = np.nan
    result["prediction_clipped"] = False
    result["low_support"] = True
    result["prediction_source"] = "unavailable"
    if frame.empty or bundle["kind"] == "unavailable":
        return result
    result["predicted_score"] = bundle["mean_score"]
    result["raw_prediction"] = bundle["mean_score"]
    result["prediction_source"] = "mean_baseline"
    if bundle["kind"] == "mean":
        return result
    feature_scope = bundle["feature_scope"]
    history = _availability_mask(frame, bundle["availability_column"])
    result.loc[~history, "prediction_source"] = (
        "mean_no_activity" if feature_scope == "retrospective" else "mean_no_history")
    if not history.any():
        return result
    values = _score_inputs(frame.loc[history], bundle["entity"], feature_scope)
    low_support = _outside_ranges(values, bundle["feature_lower"], bundle["feature_upper"])
    if bundle["kind"] == "ols":
        transformed = bundle["preprocessor"].transform(_consumer_design(values, feature_scope))
        raw = bundle["model"].predict(transformed)
        prediction = np.clip(raw, 0, 100)
        result.loc[history, "prediction_clipped"] = raw != prediction
    else:
        transformed = bundle["preprocessor"].transform(values)
        location, scale = bundle["model"].predict(transformed, return_std=True)
        prediction, score_sd, lower, upper = _logistic_normal_summary(location, scale)
        raw = prediction.copy()
        distance = bundle["neighbours"].kneighbors(transformed)[0][:, 0]
        low_support |= distance > bundle["support_distance_threshold"]
        result.loc[history, "predicted_score_sd"] = score_sd
        result.loc[history, "interval_low"] = lower
        result.loc[history, "interval_high"] = upper
        result.loc[history, "support_distance"] = distance
    result.loc[history, "predicted_score"] = prediction
    result.loc[history, "raw_prediction"] = raw
    result.loc[history, "low_support"] = low_support
    result.loc[history, "prediction_source"] = bundle["kind"]
    return result


## 13. Evaluation and final historical fits

Use whole-date 60/20/20 blocks of **observed scores**; these percentages refer to distinct dates, not row counts. Fit all preprocessing on the fitting subset. Compare OLS/GP with training-mean and median baselines using **MAE, RMSE and R²**. Merchant development data also receive two expanding-window checks.

The consumer additionally has a fixed-seed random 70/30 **entity/date** comparison using the four specified features. A daily target is never duplicated across orders before splitting. This diagnostic is not used to select features. The previously inspected periods are reused benchmarks, not fresh untouched tests; same-day features evaluate retrospective reconstruction rather than purchase-time prediction.

After evaluation, fit a final model on all supplied labels to fill historical gaps. Summaries, raw-unit consumer coefficients, kernel, holdout predictions and metrics are saved under `tables/analysis/fraud_diagnostics/`.

In [19]:
import json
import joblib
import pandas as pd
import numpy as np
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split

# Publication timestamps are absent: this is an explicit backtest assumption,
# not evidence that the supplied scores were available the following day.
SCORE_AVAILABILITY_DELAY_DAYS = 1
if SCORE_AVAILABILITY_DELAY_DAYS < 1:
    raise ValueError("Date-only data require a score-availability delay of at least one day.")


def evaluate_score_models(label_frame, entity):
    """Benchmark on whole-date blocks and compare each fitted model with constant baselines."""
    key = "user_id" if entity == "consumer" else "merchant_abn"
    dates = np.sort(label_frame.order_datetime.unique())
    if len(dates) < 5:
        raise ValueError(f"Too few scored dates for {entity} temporal evaluation.")
    validation_start = pd.Timestamp(dates[int(len(dates) * .6)])
    test_start = pd.Timestamp(dates[int(len(dates) * .8)])
    split_masks = {
        "train": label_frame.order_datetime < validation_start,
        "validation": (label_frame.order_datetime >= validation_start) & (label_frame.order_datetime < test_start),
        "test": label_frame.order_datetime >= test_start,
    }
    split_rows = []
    for name, mask in split_masks.items():
        part = label_frame.loc[mask]
        split_rows.append({"entity": entity, "split": name, "rows": len(part),
                           "entities": part[key].nunique(), "first_date": str(part.order_datetime.min().date()),
                           "last_date": str(part.order_datetime.max().date())})
    windows = [("validation", validation_start, test_start), ("test", test_start, None)]
    if entity == "merchant":
        development_dates = dates[dates < test_start]
        inner_cuts = [pd.Timestamp(development_dates[int(len(development_dates) * fraction)])
                      for fraction in [.5, .75]] + [test_start]
        windows += [(f"expanding_{i + 1}", start, end)
                    for i, (start, end) in enumerate(zip(inner_cuts[:-1], inner_cuts[1:]))]
    rows, holdouts = [], []
    for split, start, end in windows:
        available = label_frame.order_datetime + pd.Timedelta(days=SCORE_AVAILABILITY_DELAY_DAYS)
        train = label_frame.loc[available <= start]
        evaluate = label_frame.loc[(label_frame.order_datetime >= start)
                                   & (True if end is None else label_frame.order_datetime < end)]
        if train.empty or evaluate.empty:
            continue
        model = fit_score_model(train, entity)
        predictions = predict_score_model(model, evaluate)
        comparison = {
            "linear_regression" if entity == "consumer" else "gaussian_process": predictions.predicted_score,
            "training_mean": np.full(len(evaluate), train.target_score.mean()),
            "training_median": np.full(len(evaluate), train.target_score.median()),
        }
        for name, estimate in comparison.items():
            rows.append({"entity": entity, "split": split, "model": name,
                         "train_rows": len(train), "test_rows": len(evaluate),
                         "test_entities": evaluate[key].nunique(),
                         "mae": mean_absolute_error(evaluate.target_score, estimate),
                         "rmse": np.sqrt(mean_squared_error(evaluate.target_score, estimate)),
                         "r2": r2_score(evaluate.target_score, estimate)})
        record = evaluate[[key, "order_datetime", "target_score"]].join(predictions)
        record["split"] = split
        record["training_last_score_date"] = train.order_datetime.max()
        holdouts.append(record)
    return pd.DataFrame(rows), pd.DataFrame(split_rows), pd.concat(holdouts, ignore_index=True)


def evaluate_random_consumer_split(frame):
    """Fixed four-feature comparison with a random 70/30 entity/date split; no feature tuning."""
    train, test = train_test_split(frame, test_size=0.3, random_state=42)
    fitted = fit_score_model(train, "consumer")
    predicted = predict_score_model(fitted, test)
    rows = []
    estimates = {"linear_regression": predicted.predicted_score,
                 "training_mean": np.full(len(test), train.target_score.mean()),
                 "training_median": np.full(len(test), train.target_score.median())}
    for model_name, estimate in estimates.items():
        rows.append({"entity": "consumer", "split": "random_70_30_diagnostic", "model": model_name,
                     "train_rows": len(train), "test_rows": len(test), "test_entities": test.user_id.nunique(),
                     "mae": mean_absolute_error(test.target_score, estimate),
                     "rmse": np.sqrt(mean_squared_error(test.target_score, estimate)),
                     "r2": r2_score(test.target_score, estimate)})
    holdout = test[["user_id", "order_datetime", "target_score"]].join(predicted)
    holdout["split"] = "random_70_30_diagnostic"
    holdout["training_last_score_date"] = train.order_datetime.max()
    return pd.DataFrame(rows), holdout


label_frames, final_score_models = {}, {}
for entity in ["consumer", "merchant"]:
    frame = (spark.read.parquet(str(feature_paths[entity]))
             .where(F.col("target_score").isNotNull()).toPandas())
    frame["order_datetime"] = pd.to_datetime(frame.order_datetime)
    key = "user_id" if entity == "consumer" else "merchant_abn"
    frame = frame.sort_values(["order_datetime", key]).reset_index(drop=True)
    label_frames[entity] = frame
    metrics, splits, holdouts = evaluate_score_models(frame, entity)
    if entity == "consumer":
        random_metrics, random_holdout = evaluate_random_consumer_split(frame)
        metrics = pd.concat([metrics, random_metrics], ignore_index=True)
        holdouts = pd.concat([holdouts, random_holdout], ignore_index=True)
    metrics.to_csv(MODEL_OUTPUT / f"{entity}_metrics.csv", index=False)
    splits.to_csv(MODEL_OUTPUT / f"{entity}_splits.csv", index=False)
    holdouts.to_parquet(MODEL_OUTPUT / f"{entity}_holdout_predictions.parquet", index=False, compression=None)
    print(f"\n{entity.upper()} — historical score reconstruction benchmarks")
    display(splits)
    display(metrics)
    # This model fills historical gaps. Its predictions are explicitly retrospective.
    final_score_models[entity] = fit_score_model(frame, entity)
    joblib.dump(final_score_models[entity], MODEL_OUTPUT / f"{entity}_retrospective_model.joblib")
    fitted = final_score_models[entity]
    summary = {"entity": entity, "model": fitted["kind"], "labelled_rows": len(frame),
               "fitted_feature_rows": fitted["n_fit_rows"], "feature_columns": fitted["feature_columns"],
               "feature_scope": fitted["feature_scope"],
               "training_first_date": str(frame.order_datetime.min().date()),
               "training_last_date": str(frame.order_datetime.max().date()),
               "fit_scope": "retrospective_all_labels", "fit_warnings": fitted["fit_warnings"]}
    if fitted["kind"] == "ols":
        raw_coefficients = fitted["model"].coef_ / fitted["preprocessor"].scale_
        raw_intercept = fitted["model"].intercept_ - np.dot(raw_coefficients, fitted["preprocessor"].mean_)
        summary.update(design_columns=fitted["design_columns"], coefficients=fitted["model"].coef_.tolist(),
                       intercept=float(fitted["model"].intercept_), rank=int(fitted["model"].rank_),
                       raw_unit_coefficients=raw_coefficients.tolist(), raw_unit_intercept=float(raw_intercept))
    elif fitted["kind"] == "gp":
        summary["kernel"] = str(fitted["model"].kernel_)
    (MODEL_OUTPUT / f"{entity}_model_summary.json").write_text(json.dumps(summary, indent=2) + "\n")

print("Benchmarks reuse previously inspected date periods; they are not a new untouched test.")



CONSUMER — historical score reconstruction benchmarks


entity,split,rows,entities,first_date,last_date
consumer,train,9597,8228,2021-02-28,2021-10-04
consumer,validation,15035,11863,2021-10-05,2021-12-16
consumer,test,10133,8630,2021-12-17,2022-02-27


entity,split,model,train_rows,test_rows,test_entities,mae,rmse,r2
consumer,validation,linear_regression,9597,15035,11863,2.869429,4.785149,0.744827
consumer,validation,training_mean,9597,15035,11863,5.881371,9.472896,-0.000022
consumer,validation,training_median,9597,15035,11863,5.188359,10.012358,-0.117164
consumer,test,linear_regression,24632,10133,8630,1.984145,3.255715,0.885496
consumer,test,training_mean,24632,10133,8630,5.959633,9.622372,-0.000214
consumer,test,training_median,24632,10133,8630,5.263537,10.184170,-0.120418
consumer,random_70_30_diagnostic,linear_regression,24335,10430,8833,2.454960,3.825066,0.831091
consumer,random_70_30_diagnostic,training_mean,24335,10430,8833,5.937052,9.307154,-0.000017
consumer,random_70_30_diagnostic,training_median,24335,10430,8833,5.229420,9.857962,-0.121884



MERCHANT — historical score reconstruction benchmarks


entity,split,rows,entities,first_date,last_date
merchant,train,79,48,2021-03-25,2021-12-10
merchant,validation,22,16,2021-12-11,2022-01-02
merchant,test,13,9,2022-01-04,2022-02-27


entity,split,model,train_rows,test_rows,test_entities,mae,rmse,r2
merchant,validation,gaussian_process,79,22,16,7.077011,9.723114,0.678114
merchant,validation,training_mean,79,22,16,12.689032,17.163359,-0.002987
merchant,validation,training_median,79,22,16,12.751951,19.809958,-0.336159
merchant,test,gaussian_process,101,13,9,11.304290,14.261936,0.407405
merchant,test,training_mean,101,13,9,14.512487,18.560643,-0.003662
merchant,test,training_median,101,13,9,12.189001,19.818173,-0.144270
merchant,expanding_1,gaussian_process,30,49,32,12.653317,14.016202,0.042234
merchant,expanding_1,training_mean,30,49,32,15.066985,16.161264,-0.273355
merchant,expanding_1,training_median,30,49,32,12.119195,14.488077,-0.023341
merchant,expanding_2,gaussian_process,79,22,16,7.077011,9.723114,0.678114


Benchmarks reuse previously inspected date periods; they are not a new untouched test.


## 14. Historical imputation and separate earlier-history estimates

For the current historical-scoring objective, use **`effective_score`**: preserve every supplied score and substitute the final retrospective model only when it is missing. `retrospective_estimated_score` always contains that model estimate. Completed same-day activity is an intended input, so these are not estimates that were available at the beginning of the day.

The previously added `purchase_time_score` is retained separately. Its consumer model uses previous-calendar-day activity; its merchant model uses earlier-day history. Monthly walk-forward fits use only scores assumed available at the start of the month, with the model month and last training-score date recorded. Publication timestamps are absent: `SCORE_AVAILABILITY_DELAY_DAYS = 1` remains an explicit, unverified availability assumption. Earliest months without earlier labels stay NULL.

All transactions sharing an entity/date receive the same entity score. The same-day retrospective model is never used to generate the `purchase_time_score` column.

In [20]:
import pyarrow as pa
import pyarrow.dataset as ds
import pyarrow.parquet as pq
from tempfile import TemporaryDirectory


def write_score_predictions(entity, feature_path, labels, final_model):
    """Save final-fit imputation alongside monthly walk-forward predictions with explicit cutoffs."""
    key = "user_id" if entity == "consumer" else "merchant_abn"
    dataset = ds.dataset(str(feature_path), format="parquet")
    models_by_month = {}
    model_records = []
    counts = {"rows": 0, "observed": 0, "imputed": 0, "purchase_time_unavailable": 0,
              "retrospective_clipped": 0, "retrospective_low_support": 0}
    path = ANALYSIS / f"{entity}_score_predictions.parquet"
    schema = pa.schema([
        (key, pa.int64()), ("order_datetime", pa.date32()), ("observed_score", pa.float64()),
        ("retrospective_estimated_score", pa.float64()), ("effective_score", pa.float64()),
        ("score_source", pa.string()), ("retrospective_raw_prediction", pa.float64()),
        ("retrospective_prediction_source", pa.string()),
        ("retrospective_interval_low", pa.float64()), ("retrospective_interval_high", pa.float64()),
        ("retrospective_clipped", pa.bool_()), ("retrospective_low_support", pa.bool_()),
        ("purchase_time_score", pa.float64()), ("purchase_time_interval_low", pa.float64()),
        ("purchase_time_interval_high", pa.float64()), ("purchase_time_low_support", pa.bool_()),
        ("purchase_time_available", pa.bool_()), ("purchase_time_training_end", pa.date32()),
        ("purchase_time_prediction_source", pa.string()),
        ("purchase_time_model_month", pa.date32()), ("score_availability_delay_days", pa.int32()),
    ])
    with TemporaryDirectory(prefix=f".{entity}_scoring_", dir=ANALYSIS) as work:
        temporary = Path(work) / path.name
        with pq.ParquetWriter(temporary, schema, compression=None) as writer:
            for batch in dataset.scanner(batch_size=32768).to_batches():
                frame = batch.to_pandas()
                frame["order_datetime"] = pd.to_datetime(frame.order_datetime)
                retrospective = predict_score_model(final_model, frame)
                result = pd.DataFrame({key: frame[key], "order_datetime": frame.order_datetime,
                                       "observed_score": frame.target_score})
                result["retrospective_estimated_score"] = retrospective.predicted_score
                result["effective_score"] = frame.target_score.fillna(retrospective.predicted_score)
                result["score_source"] = np.where(frame.target_score.notna(), "observed", "retrospective_model")
                result["retrospective_prediction_source"] = retrospective.prediction_source
                for output, source in [("raw_prediction", "raw_prediction"), ("interval_low", "interval_low"),
                                       ("interval_high", "interval_high"), ("clipped", "prediction_clipped"),
                                       ("low_support", "low_support")]:
                    result[f"retrospective_{output}"] = retrospective[source]
                result["purchase_time_score"] = np.nan
                result["purchase_time_interval_low"] = np.nan
                result["purchase_time_interval_high"] = np.nan
                result["purchase_time_low_support"] = True
                result["purchase_time_prediction_source"] = "unavailable"
                result["purchase_time_training_end"] = pd.NaT
                result["purchase_time_model_month"] = frame.order_datetime.dt.to_period("M").dt.to_timestamp()
                result["score_availability_delay_days"] = SCORE_AVAILABILITY_DELAY_DAYS
                for month, indices in result.groupby("purchase_time_model_month").groups.items():
                    if month not in models_by_month:
                        available = labels.order_datetime + pd.Timedelta(days=SCORE_AVAILABILITY_DELAY_DAYS)
                        training = labels.loc[available <= month]
                        bundle = fit_score_model(training, entity, feature_scope="purchase_time")
                        training_end = training.order_datetime.max()
                        models_by_month[month] = (bundle, training_end)
                        model_records.append({"month": str(month.date()), "training_rows": len(training),
                                              "feature_scope": bundle["feature_scope"],
                                              "model_kind": bundle["kind"],
                                              "training_end": None if pd.isna(training_end) else str(training_end.date())})
                    bundle, training_end = models_by_month[month]
                    prediction = predict_score_model(bundle, frame.loc[indices])
                    result.loc[indices, "purchase_time_score"] = prediction.predicted_score
                    result.loc[indices, "purchase_time_interval_low"] = prediction.interval_low
                    result.loc[indices, "purchase_time_interval_high"] = prediction.interval_high
                    result.loc[indices, "purchase_time_low_support"] = prediction.low_support
                    result.loc[indices, "purchase_time_prediction_source"] = prediction.prediction_source
                    result.loc[indices, "purchase_time_training_end"] = training_end
                result["purchase_time_available"] = result.purchase_time_score.notna()
                available_rows = result.purchase_time_available
                if not ((result.loc[available_rows, "purchase_time_training_end"]
                         + pd.Timedelta(days=SCORE_AVAILABILITY_DELAY_DAYS))
                        <= result.loc[available_rows, "purchase_time_model_month"]).all():
                    raise AssertionError("Walk-forward model used scores unavailable at its cutoff.")
                if result.effective_score.isna().any() or not result.effective_score.between(0, 100).all():
                    raise AssertionError("Historical imputation did not produce finite bounded scores.")
                observed = frame.target_score.notna()
                if not np.array_equal(result.loc[observed, "effective_score"], frame.loc[observed, "target_score"]):
                    raise AssertionError("A supplied score was changed.")
                writer.write_table(pa.Table.from_pandas(result, schema=schema, preserve_index=False))
                counts["rows"] += len(result)
                counts["observed"] += int(observed.sum())
                counts["imputed"] += int((~observed).sum())
                counts["purchase_time_unavailable"] += int((~available_rows).sum())
                counts["retrospective_clipped"] += int(result.retrospective_clipped.sum())
                counts["retrospective_low_support"] += int(result.retrospective_low_support.sum())
                if counts["rows"] % (32768 * 32) < 32768:
                    print(f"{entity}: scored {counts['rows']:,} entity/dates", flush=True)
        temporary.replace(path)
    pd.DataFrame(model_records).sort_values("month").to_csv(
        MODEL_OUTPUT / f"{entity}_walk_forward_models.csv", index=False)
    return path, counts


prediction_paths, prediction_audit = {}, {}
for entity in ["consumer", "merchant"]:
    prediction_paths[entity], prediction_audit[entity] = write_score_predictions(
        entity, feature_paths[entity], label_frames[entity], final_score_models[entity])
    print(entity, prediction_audit[entity])


consumer: scored 1,048,576 entity/dates
consumer: scored 2,105,097 entity/dates
consumer: scored 3,162,444 entity/dates
consumer: scored 4,219,391 entity/dates
consumer: scored 5,274,169 entity/dates
consumer: scored 6,297,184 entity/dates
consumer: scored 7,354,308 entity/dates
consumer: scored 8,412,028 entity/dates
consumer {'rows': 8976957, 'observed': 34765, 'imputed': 8942192, 'purchase_time_unavailable': 12259, 'retrospective_clipped': 2, 'retrospective_low_support': 8742685}
merchant: scored 1,056,689 entity/dates
merchant {'rows': 1344515, 'observed': 114, 'imputed': 1344401, 'purchase_time_unavailable': 65125, 'retrospective_clipped': 0, 'retrospective_low_support': 1218986}


## 15. Join daily estimates to transactions and audit preservation

Outputs: `tables/analysis/transactions_with_score_estimates` and `merchant_score_summary`. Original `consumer_fraud_probability` / `merchant_fraud_probability` columns retain their NULLs. **`effective_consumer_fraud_probability` and `effective_merchant_fraud_probability` are the completed historical scores to use for the current analysis.** The audit checks row preservation, source-score equality, complete bounded historical scores and prediction joins.

The output preserves a daily score for each entity. The existing merchant summary is a transaction-weighted average of those daily scores; it is not an independently supplied overall fraud probability. No overall customer score is created.

The analysis transaction table retains the average spend across strictly earlier days, signed `spending_diff`, and `absolute_spending_diff`. This stage does not infer `is_fraud` labels. Merchant revenue is calculated from the completed export in section 17.

For ordinary downstream analysis, use `tables/analysis/transactions_imputed/transactions_imputed.parquet`, created in section 16, with the conventional `consumer_fraud_probability` and `merchant_fraud_probability` names. This internal table retains `effective_*`, original-score columns and `purchase_time_*` diagnostics; the UMAP intentionally reads the original nullable curated scores.

In [21]:
def attach_score_predictions(orders, consumer_predictions, merchant_predictions, consumer_features):
    """Join one score row per entity/date while retaining supplied scores unchanged."""
    result = orders.drop("consumer_avg_spending", "spending_diff", "absolute_spending_diff")
    for entity, predictions in [("consumer", consumer_predictions), ("merchant", merchant_predictions)]:
        key = "user_id" if entity == "consumer" else "merchant_abn"
        names = {
            "observed_score": f"{entity}_observed_score_check",
            "retrospective_estimated_score": f"estimated_{entity}_fraud_probability",
            "effective_score": f"effective_{entity}_fraud_probability",
            "purchase_time_score": f"purchase_time_{entity}_fraud_probability",
        }
        renamed = predictions.select(key, "order_datetime", *[
            F.col(c).alias(names.get(c, f"{entity}_{c}"))
            for c in predictions.columns if c not in [key, "order_datetime"]
        ]).withColumn(f"{entity}_prediction_present", F.lit(True))
        result = result.join(renamed, [key, "order_datetime"], "left")
    history = consumer_features.select("user_id", "order_datetime",
                                        F.col("prior_avg_spend").alias("consumer_avg_spending"))
    return (result.join(history, ["user_id", "order_datetime"], "left")
            .withColumn("spending_diff", F.col("dollar_value") - F.col("consumer_avg_spending"))
            .withColumn("absolute_spending_diff", F.abs("spending_diff")))


curated_orders = spark.read.parquet(str(root / "tables/curated/transactions"))
prediction_tables = {entity: spark.read.parquet(str(path)) for entity, path in prediction_paths.items()}
consumer_history = spark.read.parquet(str(feature_paths["consumer"]))
scored_orders = attach_score_predictions(curated_orders, prediction_tables["consumer"],
                                        prediction_tables["merchant"], consumer_history)
source_rows = curated_orders.count()
audit_fields = [F.count("*").alias("transaction_rows")]
for entity in ["consumer", "merchant"]:
    observed = F.col(f"{entity}_fraud_probability")
    effective = F.col(f"effective_{entity}_fraud_probability")
    audit_fields.extend([
        F.sum(F.col(f"{entity}_prediction_present").isNull().cast("long")).alias(f"{entity}_missing_joins"),
        F.sum((~observed.eqNullSafe(F.col(f"{entity}_observed_score_check"))).cast("long")).alias(f"{entity}_source_mismatches"),
        F.sum((observed.isNotNull() & ~observed.eqNullSafe(effective)).cast("long")).alias(f"{entity}_observed_changes"),
        F.sum((effective.isNull() | F.isnan(effective) | ~effective.between(0, 100)).cast("long")).alias(f"{entity}_invalid_effective_scores"),
        F.sum(F.col(f"purchase_time_{entity}_fraud_probability").isNull().cast("long")).alias(f"{entity}_purchase_time_unavailable"),
    ])
transaction_audit = scored_orders.agg(*audit_fields).first().asDict()
if transaction_audit["transaction_rows"] != source_rows:
    raise AssertionError("Score joins changed the number of transactions.")
for key, value in transaction_audit.items():
    if key.endswith(("missing_joins", "source_mismatches", "observed_changes", "invalid_effective_scores")) and value:
        raise AssertionError(f"Fraud imputation audit failed: {key}={value}")

scored_orders = scored_orders.drop("consumer_observed_score_check", "merchant_observed_score_check",
                                    "consumer_prediction_present", "merchant_prediction_present")
transaction_output = ANALYSIS / "transactions_with_score_estimates"
saved_scores = write_table(scored_orders, transaction_output, "order_month")

# Merchant summaries are retrospective descriptions; score coverage remains visible.
merchant_score_summary = saved_scores.groupBy("merchant_abn").agg(
    F.count("*").alias("scored_transaction_count"),
    F.avg("consumer_fraud_probability").alias("avg_observed_consumer_fraud_score"),
    F.avg("merchant_fraud_probability").alias("avg_observed_merchant_fraud_score"),
    F.avg("effective_consumer_fraud_probability").alias("avg_effective_consumer_fraud_score"),
    F.avg("effective_merchant_fraud_probability").alias("avg_effective_merchant_fraud_score"),
    F.avg(F.col("consumer_fraud_probability").isNotNull().cast("double")).alias("consumer_observed_score_coverage"),
    F.avg(F.col("merchant_fraud_probability").isNotNull().cast("double")).alias("merchant_observed_score_coverage"))
merchant_output = ANALYSIS / "merchant_score_summary"
write_table(merchant_score_summary, merchant_output)

audit = {
    "source_transaction_rows": source_rows, "entity_date_predictions": prediction_audit,
    "transaction_checks": transaction_audit,
    "score_units": "Supplied 0–100 scores, not calibrated transaction fraud outcome probabilities.",
    "historical_imputation": "effective_* preserves supplied scores and fills gaps using same-day activity and a final-fit retrospective model.",
    "score_granularity": "One score per customer/date or merchant/date; shared by transactions on that entity/date.",
    "consumer_retrospective_features": ["same_day_spending", "same_day_average_order_value", "same_day_spending_squared", "same_day_order_count_squared"],
    "merchant_retrospective_features": final_score_models["merchant"]["feature_columns"],
    "purchase_time_policy": "Monthly model fitted only to scores assumed available by the first of the month; NULL before any earlier labels exist.",
    "score_availability_delay_days": SCORE_AVAILABILITY_DELAY_DAYS,
    "availability_limitation": "Score publication timestamps are absent. The configured delay is an assumption, not verified historical availability.",
    "history_policy": "Retrospective models use completed same-day activity. Separate purchase_time models retain previous-day consumer and earlier-date merchant features.",
    "outputs": {"transactions": str(transaction_output), "merchant_summary": str(merchant_output)},
}
(MODEL_OUTPUT / "imputation_audit.json").write_text(json.dumps(audit, indent=2) + "\n")
display(pd.DataFrame([transaction_audit]))
print(f"Saved transaction score estimates: {transaction_output}")
print("Use effective_* for the current historical scoring objective; scores remain at entity/date level.")


transaction_rows,consumer_missing_joins,consumer_source_mismatches,consumer_observed_changes,consumer_invalid_effective_scores,consumer_purchase_time_unavailable,merchant_missing_joins,merchant_source_mismatches,merchant_observed_changes,merchant_invalid_effective_scores,merchant_purchase_time_unavailable
14195505,0,0,0,0,17107,0,0,0,0,563440


Saved transaction score estimates: /Users/trollinmaster/Documents/Uni/Coding/VSCode/MAST30034/industry-project-group-19/tables/analysis/transactions_with_score_estimates
Use effective_* for the current historical scoring objective; scores remain at entity/date level.


## 16. Final completed transaction dataset

The final deliverables are **`tables/analysis/transactions_imputed/transactions_imputed.csv`** and **`tables/analysis/transactions_imputed/transactions_imputed.parquet`**. They contain identical data, with one row per order and no missing values across the exported columns. Prefer Parquet when loading the data because it preserves types, including postcode strings.

Use the core transaction fields and completed fraud scores under their usual names, `consumer_fraud_probability` and `merchant_fraud_probability`. Supplied scores are preserved; missing scores use the existing all-label consumer OLS and merchant Gaussian-process estimates. These remain retrospective entity/date scores on the supplied 0–100 scale. The export also retains historical average spending, signed/absolute spending differences and order month.

For orders without previous consumer spending history, fill `consumer_avg_spending` with the median available historical average across transaction rows, then recompute `spending_diff` and `absolute_spending_diff`. All other exported fields are checked for completeness. This fallback is for the completed historical dataset.

Keep three provenance fields: `consumer_score_source`, `merchant_score_source` and `consumer_avg_spending_imputed`. The score-source fields identify `observed` versus `retrospective_model` values. Model internals, evaluation tables and figure files live under `tables/analysis/fraud_diagnostics/`; the original nullable data remain available to the missingness analysis.

This section can run independently after section 15, using DuckDB without Spark. It stages both files, checks row/key preservation, complete finite values, bounded scores and matching fingerprints across all columns, then publishes the pair. Its detailed audit is saved internally with the diagnostics.


In [22]:
from pathlib import Path
from tempfile import TemporaryDirectory
import json
import math
import duckdb


def export_imputed_transactions(project_root):
    """Export one completed historical transaction dataset in CSV and Parquet."""
    project_root = Path(project_root).resolve()
    output = project_root / "tables/analysis/transactions_imputed"
    output.mkdir(parents=True, exist_ok=True)
    diagnostics = project_root / "tables/analysis/fraud_diagnostics"
    diagnostics.mkdir(parents=True, exist_ok=True)
    source = project_root / "tables/analysis/transactions_with_score_estimates"
    if not list(source.rglob("*.parquet")):
        raise FileNotFoundError("Run ETL sections 11–15 before exporting completed transactions.")
    paths = {kind: output / f"transactions_imputed.{kind}" for kind in ["csv", "parquet"]}

    # Stage both deliverables and validate them before replacing a previous export.
    with TemporaryDirectory(prefix=".transactions-export-", dir=output) as scratch, duckdb.connect() as con:
        con.execute("SET memory_limit='1GB'")
        con.execute("SET threads=4")
        con.execute("SET temp_directory=?", [str(Path(scratch) / "duckdb")])
        con.read_parquet(str(source / "**/*.parquet"), hive_partitioning=False).create_view("score_details")
        original = con.execute("""SELECT count(*) AS rows,
            count(*) FILTER (WHERE consumer_fraud_probability IS NULL) AS consumer_scores_imputed,
            count(*) FILTER (WHERE merchant_fraud_probability IS NULL) AS merchant_scores_imputed,
            count(*) FILTER (WHERE consumer_avg_spending IS NULL) AS spending_history_imputed,
            median(consumer_avg_spending) AS spending_history_median
            FROM score_details""").fetchdf().iloc[0]
        median_average = float(original.spending_history_median)
        if not int(original.rows) or not math.isfinite(median_average):
            raise ValueError("A nonempty dataset and finite observed spending median are required.")
        con.execute("CREATE TEMP TABLE spending_fallback AS SELECT CAST(? AS DOUBLE) AS median_average", [median_average])
        con.execute("""CREATE TEMP VIEW completed_transactions AS
            WITH completed AS (
                SELECT order_id, order_datetime, dollar_value, user_id, merchant_abn,
                    coalesce(consumer_fraud_probability, effective_consumer_fraud_probability)
                        AS consumer_fraud_probability,
                    coalesce(merchant_fraud_probability, effective_merchant_fraud_probability)
                        AS merchant_fraud_probability,
                    consumer_postcode, consumer_gender, consumer_district_income,
                    income_matched, consumer_record_matched, merchant_record_matched,
                    coalesce(consumer_avg_spending, median_average) AS consumer_avg_spending,
                    consumer_score_source, merchant_score_source,
                    consumer_avg_spending IS NULL AS consumer_avg_spending_imputed
                FROM score_details CROSS JOIN spending_fallback)
            SELECT order_id, order_datetime, dollar_value, user_id, merchant_abn,
                consumer_fraud_probability, merchant_fraud_probability,
                consumer_postcode, consumer_gender, consumer_district_income,
                income_matched, consumer_record_matched, merchant_record_matched,
                consumer_avg_spending,
                dollar_value - consumer_avg_spending AS spending_diff,
                abs(dollar_value - consumer_avg_spending) AS absolute_spending_diff,
                strftime(order_datetime, '%Y-%m') AS order_month,
                consumer_score_source, merchant_score_source, consumer_avg_spending_imputed
            FROM completed""")
        schema = {row[0]: row[1] for row in con.execute("DESCRIBE completed_transactions").fetchall()}
        columns = list(schema)
        quoted = [f'"{column}"' for column in columns]
        null_checks = ', '.join(f'count(*) FILTER (WHERE {c} IS NULL) AS {c}' for c in quoted)
        numeric = [column for column, kind in schema.items() if kind in ["DOUBLE", "FLOAT"]]
        invalid_numbers = ' OR '.join(f'NOT isfinite("{c}")' for c in numeric)
        invalid_scores = """consumer_fraud_probability NOT BETWEEN 0 AND 100
                            OR merchant_fraud_probability NOT BETWEEN 0 AND 100"""

        def inspect_view(view):
            row = con.execute(f"SELECT {null_checks} FROM {view}").fetchone()
            null_counts = dict(zip(columns, map(int, row)))
            if any(null_counts.values()):
                raise AssertionError(f"Completed dataset still has NULLs: {null_counts}")
            invalid = con.execute(f"SELECT count(*) FROM {view} WHERE ({invalid_numbers}) OR ({invalid_scores})").fetchone()[0]
            if invalid:
                raise AssertionError(f"Completed dataset has {invalid} invalid numerical values.")
            # Two order-independent row fingerprints compare all values in both formats.
            hashed = 'hash(' + ', '.join(quoted) + ')'
            signature = con.execute(f"""SELECT count(*),
                sum(CAST({hashed} AS HUGEINT)), bit_xor({hashed}) FROM {view}""").fetchone()
            return dict(rows=int(signature[0]), checksum_sum=int(signature[1]),
                        checksum_xor=int(signature[2]), null_counts=null_counts)

        print("Validating completed transaction values...", flush=True)
        expected = inspect_view("completed_transactions")
        if expected["rows"] != int(original.rows):
            raise AssertionError("Export preparation changed the transaction count.")
        staged_parquet = Path(scratch) / "transactions_imputed.parquet"
        staged_csv = Path(scratch) / "transactions_imputed.csv"
        print("Writing the completed Parquet...", flush=True)
        con.execute("COPY completed_transactions TO ? (FORMAT PARQUET, COMPRESSION ZSTD)", [str(staged_parquet)])
        con.read_parquet(str(staged_parquet)).create_view("exported_parquet")
        print("Writing the completed CSV...", flush=True)
        con.execute("COPY exported_parquet TO ? (FORMAT CSV, HEADER TRUE, FORCE_QUOTE *)", [str(staged_csv)])
        # Explicit types preserve postcode strings and compare floating-point values exactly.
        con.read_csv(str(staged_csv), columns=schema, header=True, auto_detect=False).create_view("exported_csv")
        for view in ["exported_parquet", "exported_csv"]:
            print(f"Checking every row in {view}...", flush=True)
            if inspect_view(view) != expected:
                raise AssertionError(f"{view} does not match the completed dataset.")
        unique_orders = int(con.execute("SELECT count(DISTINCT order_id) FROM exported_parquet").fetchone()[0])
        if unique_orders != expected["rows"]:
            raise AssertionError("Final data must contain exactly one row per order_id.")
        observed_changes = con.execute("""SELECT
            count(*) FILTER (WHERE consumer_fraud_probability IS NOT NULL AND
                consumer_fraud_probability IS DISTINCT FROM effective_consumer_fraud_probability),
            count(*) FILTER (WHERE merchant_fraud_probability IS NOT NULL AND
                merchant_fraud_probability IS DISTINCT FROM effective_merchant_fraud_probability)
            FROM score_details""").fetchone()
        if any(observed_changes):
            raise AssertionError("The internal completed scores altered a supplied fraud score.")
        staged_parquet.replace(paths["parquet"])
        staged_csv.replace(paths["csv"])

    audit = {
        "rows": expected["rows"], "columns": columns, "schema": schema,
        "unique_orders": unique_orders, "null_counts": expected["null_counts"],
        "consumer_scores_imputed": int(original.consumer_scores_imputed),
        "merchant_scores_imputed": int(original.merchant_scores_imputed),
        "spending_history_imputed": int(original.spending_history_imputed),
        "spending_history_median": median_average,
        "spending_imputation": "Median of available historical consumer averages across transaction rows; recompute signed and absolute spending differences.",
        "fraud_imputation": "Preserve supplied scores; fill missing consumer scores with final OLS and merchant scores with final Matérn-nu=2.5 GP historical estimates.",
        "scope": "Completed historical transaction dataset; scores remain on the supplied 0–100 scale at entity/date granularity.",
        "validation": "No NULL/nonfinite values; one row per order; supplied scores preserved; both formats match row counts and two fingerprints over every column.",
        "paths": {key: str(value) for key, value in paths.items()},
        "file_bytes": {key: value.stat().st_size for key, value in paths.items()},
    }
    (diagnostics / "completed_transactions_export_audit.json").write_text(json.dumps(audit, indent=2, allow_nan=False) + "\n")
    return audit


In [23]:
from IPython.display import display
import pandas as pd

export_root = next((p for p in [Path.cwd(), *Path.cwd().parents]
                    if (p / "tables/analysis/transactions_with_score_estimates").is_dir()), None)
if export_root is None:
    raise FileNotFoundError("Run from industry-project-group-19 or its code directory after ETL section 15.")
completed_export = export_imputed_transactions(export_root)
print(f"Completed dataset: {completed_export['rows']:,} orders, {len(completed_export['columns'])} columns, zero missing values.")
print(f"Fraud values imputed: {completed_export['consumer_scores_imputed']:,} consumer; {completed_export['merchant_scores_imputed']:,} merchant.")
print(f"Historical averages imputed: {completed_export['spending_history_imputed']:,}; median fallback ${completed_export['spending_history_median']:.6f}.")
display(pd.DataFrame([{"format": kind.upper(), "file": path,
                       "size_MB": round(completed_export["file_bytes"][kind] / 1_000_000, 1)}
                     for kind, path in completed_export["paths"].items()]))
print("Validation passed: unique orders, complete values, supplied scores preserved, CSV and Parquet contents match.")


Validating completed transaction values...
Writing the completed Parquet...
Writing the completed CSV...
Checking every row in exported_parquet...
Checking every row in exported_csv...
Completed dataset: 14,195,505 orders, 20 columns, zero missing values.
Fraud values imputed: 14,115,157 consumer; 14,191,446 merchant.
Historical averages imputed: 33,287; median fallback $160.202155.


format,file,size_MB
CSV,/Users/trollinmaster/Documents/Uni/Coding/VSCode/MAST30034/industry-project-group-19/tables/analysis/transactions_imputed/transactions_imputed.csv,4408.4
PARQUET,/Users/trollinmaster/Documents/Uni/Coding/VSCode/MAST30034/industry-project-group-19/tables/analysis/transactions_imputed/transactions_imputed.parquet,909.9


Validation passed: unique orders, complete values, supplied scores preserved, CSV and Parquet contents match.


## 17. Merchant metrics from completed transactions

Run after section 16. Aggregate `tables/analysis/transactions_imputed/transactions_imputed.parquet` so merchant averages include supplied and imputed scores. `take_rate` and fraud scores stay in their supplied percentage units; divide each by 100 when computing the fee-revenue proxy:

`expected_revenue = (take_rate / 100) × sum(dollar_value × (1 − consumer_fraud_probability / 100))`.

This discounts each order by its consumer score before summing, assuming fees are earned on nonfraudulent transaction value. It is a modelling proxy based on entity/date scores. The output keeps one row per merchant master record. Orders with an unknown merchant remain in the completed transaction dataset and are reported separately; no take rate is invented for them. A master merchant with no orders has zero totals/revenue and undefined average scores.

Save to `tables/curated/merchants` using the staged writer. The audit verifies row preservation, expected revenue between zero and gross fee revenue, and reconciliation of matched/unmatched order counts.


In [24]:
def build_enriched_merchants(completed_orders, merchant_base):
    """Aggregate completed daily scores and discount each order's fee by consumer fraud risk."""
    metric_columns = ["avg_merchant_fraud_prob", "avg_consumer_district_income",
                      "avg_consumer_fraud_prob", "total_revenue", "total_transactions",
                      "avg_value_per_transaction", "expected_revenue", "_fraud_adjusted_value"]
    base = merchant_base.drop(*metric_columns)
    merchant_metrics = completed_orders.groupBy("merchant_abn").agg(
        F.avg("merchant_fraud_probability").alias("avg_merchant_fraud_prob"),
        F.avg("consumer_district_income").alias("avg_consumer_district_income"),
        F.avg("consumer_fraud_probability").alias("avg_consumer_fraud_prob"),
        F.sum("dollar_value").alias("total_revenue"),
        F.count(F.lit(1)).alias("total_transactions"),
        F.sum(F.col("dollar_value") * (1 - F.col("consumer_fraud_probability") / 100.0))
            .alias("_fraud_adjusted_value"),
    )
    return (base.join(merchant_metrics, "merchant_abn", "left")
            .fillna(0, subset=["total_revenue", "total_transactions", "_fraud_adjusted_value"])
            .withColumn("avg_value_per_transaction", F.when(F.col("total_transactions") > 0,
                        F.col("total_revenue") / F.col("total_transactions")))
            .withColumn("expected_revenue", (F.col("take_rate") / 100.0) * F.col("_fraud_adjusted_value"))
            .drop("_fraud_adjusted_value"))


transactions_path = ROOT / "tables/analysis/transactions_imputed/transactions_imputed.parquet"
merchants_path = CURATED / "merchants"
if not transactions_path.is_file():
    raise FileNotFoundError("Run section 16 before calculating merchant metrics.")
spark.catalog.refreshByPath(str(transactions_path))
transactions_for_merchants = spark.read.parquet(str(transactions_path))
merchants_to_enrich = spark.read.parquet(str(merchants_path))
require_key(merchants_to_enrich, ["merchant_abn"], "merchant master")
invalid_take_rate = (F.col("take_rate").isNull() | F.isnan("take_rate")
                     | ~F.col("take_rate").between(0, 100))
if merchants_to_enrich.where(invalid_take_rate).limit(1).count():
    raise ValueError("Merchant take rates must be finite percentages between 0 and 100.")
enriched_merchants = build_enriched_merchants(transactions_for_merchants, merchants_to_enrich)
merchant_total = merchants_to_enrich.count()
merchant_checks = enriched_merchants.agg(
    F.count("*").alias("merchant_rows"),
    F.sum(F.col("total_transactions")).alias("matched_transaction_rows"),
    F.sum(F.col("expected_revenue")).alias("total_expected_fee_revenue"),
    F.sum((F.col("expected_revenue").isNull() | F.isnan("expected_revenue")
           | (F.col("expected_revenue") < -1e-8)
           | (F.col("expected_revenue") > F.col("total_revenue") * F.col("take_rate") / 100.0 + 1e-8))
          .cast("long")).alias("invalid_expected_revenue"),
).first().asDict()
if merchant_checks["merchant_rows"] != merchant_total or merchant_checks["invalid_expected_revenue"]:
    raise AssertionError(f"Merchant enrichment failed its row/value checks: {merchant_checks}")
missing_master_orders = transactions_for_merchants.join(
    merchants_to_enrich.select("merchant_abn"), "merchant_abn", "left_anti")
unmatched = missing_master_orders.agg(F.count("*").alias("unmatched_transaction_rows"),
                                     F.countDistinct("merchant_abn").alias("unmatched_merchants")).first().asDict()
merchant_checks.update(unmatched)
if int(merchant_checks["matched_transaction_rows"] or 0) + unmatched["unmatched_transaction_rows"] != completed_export["rows"]:
    raise AssertionError("Merchant totals do not reconcile to the completed transaction dataset.")
enriched_merchants = write_table(enriched_merchants, merchants_path)
(MODEL_OUTPUT / "merchant_enrichment_audit.json").write_text(json.dumps(merchant_checks, indent=2) + "\n")
display(pd.DataFrame([merchant_checks]))
print("Saved merchant metrics based on completed scores:", merchants_path)
print("Take rate and fraud scores are percentages; expected_revenue discounts each transaction before aggregation.")


merchant_rows,matched_transaction_rows,total_expected_fee_revenue,invalid_expected_revenue,unmatched_transaction_rows,unmatched_merchants
4026,13614675,8.921421e+07,0,580830,396


Saved merchant metrics based on completed scores: /Users/trollinmaster/Documents/Uni/Coding/VSCode/MAST30034/industry-project-group-19/tables/curated/merchants
Take rate and fraud scores are percentages; expected_revenue discounts each transaction before aggregation.


In [25]:
# Read the whole saved merchant table, not an arbitrary part file.
spark.read.parquet(str(CURATED / "merchants")).orderBy("merchant_abn").show(5, truncate=False)


+------------+------------------------------------+-----------------------------------------------------------------------------------------------------------------+-------------------------------------------------------------------------------------+------------+---------+-----------------------+----------------------------+-----------------------+------------------+------------------+-------------------------+------------------+
|merchant_abn|name                                |tags                                                                                                             |category                                                                             |letter_group|take_rate|avg_merchant_fraud_prob|avg_consumer_district_income|avg_consumer_fraud_prob|total_revenue     |total_transactions|avg_value_per_transaction|expected_revenue  |
+------------+------------------------------------+-------------------------------------------------------------------------------